# Download data

This notebook downloads the raw data for the methylation studies.

If we had more time, we would reprocess the data with minfi, to ensure all the datasets are processed in the same way.

In [0]:
import os
import urllib.request

# Define dataset download URLs
raw_data_urls = {
    "GSE213478": "https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE213478&format=file",
    "GSE289137": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE289nnn/GSE289137/suppl/GSE289137%5FBetaValues.csv.gz"
}

# Define output path in Databricks DBFS volume
output_base_path = "/Volumes/bronze/methylation/geo_datasets"

# Ensure local mount exists (Databricks handles DBFS paths through /dbfs)
dbutils.fs.mkdirs(output_base_path)

# Loop through each dataset
for accession, url in raw_data_urls.items():
    print(f"Downloading {accession} from {url} ...")
    file_ext = url.split("file=")[-1] if "file=" in url else f"{accession}.tar"
    output_file = os.path.join(output_base_path, accession, file_ext)
    
    try:
        urllib.request.urlretrieve(url, output_file)
        print(f"Saved {accession} to {output_file}")
    except Exception as e:
        print(f"Failed to download {accession}: {e}")


In [0]:
%sh
mkdir -p /Volumes/bronze/methylation/geo_datasets/GSE213478
tar -xvf /Volumes/bronze/methylation/geo_datasets/GSE213478_RAW.tar -C /Volumes/bronze/methylation/geo_datasets/GSE213478
gunzip /Volumes/bronze/methylation/geo_datasets/GSE213478/GPL21145_MethylationEPIC_15073387_v-1-0.csv.gz

In [0]:
!ls /Volumes/bronze/methylation/geo_datasets

In [0]:
%sh
mkdir -p /Volumes/bronze/methylation/geo_datasets
cd /Volumes/bronze/methylation/geo_datasets

wget "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE213nnn/GSE213478/suppl/GSE213478%5Fmethylation%5FDNAm%5Fnoob%5Ffinal%5FBMIQ%5Fall%5Ftissues%5F987.txt.gz"


In [0]:
!wget "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE213nnn/GSE213478/suppl/GSE213478%5Fmethylation%5FDNAm%5Fnoob%5Ffinal%5FBMIQ%5Fall%5Ftissues%5F987.txt.gz"

## Download CpG Corpus from CpGPT

See instructions here: https://github.com/lcamillo/CpGPT/blob/main/README.md

In [0]:
!aws s3 sync s3://cpgpt-lucascamillo-public/data/cpgcorpus/raw /Volumes/bronze/methylation/cpgcorpus/raw --request-payer requester


In [0]:
source_path = "s3://cpgpt-lucascamillo-public/data/cpgcorpus/raw/"
target_path = "dbfs:/mnt/bronze/methylation/cpgcorpus/raw/"

# List files in the source path
files = dbutils.fs.ls(source_path)

for f in files:
    if not f.isDir():  # skip subdirectories unless you handle them recursively
        print(f"Copying {f.path}")
        dbutils.fs.cp(f.path, target_path + f.name)
